# RF-DETR: Mammography Lesion Detection Quickstart

This notebook demonstrates the end-to-end workflow for RF-DETR on Google Colab:
1. Automated environment setup & compilation of Detrex CUDA operators
2. Automated test verification
3. Dataset annotation visualization
4. Training with Detrex DINO & RF-DETR MultiScaleProjector
5. TensorBoard monitoring
6. Standalone COCO evaluation
7. Side-by-side inference visualization (Ground Truth vs Model Predictions)

### Hardware Requirement:
- GPU Runtime: NVIDIA T4, V100, A100, or L4 (Compute capability >= 7.0)

In [ ]:
# Step 1: Environment Setup & Google Drive Mount
# 1a. Mount Google Drive (if datasets or checkpoints are stored on Drive)
from google.colab import drive
drive.mount('/content/drive')

# 1b. Automated Setup & Compilation
# Clones Detrex, installs Detectron2, compiles MSDeformAttn, creates symlink, and installs RF-DETR
!bash scripts/setup_colab.sh


In [ ]:
# Step 2: Verification Test Suite
# Runs automated offline tests on synthetic data (85 tests)
!python -m pytest tests/ -v


In [ ]:
# Step 3: Dataset Paths Configuration
import os

DATA_DIR = "/content/mammo_data/images"
TRAIN_JSON = "/content/drive/MyDrive/EMBED_Dataset/curated/full_dataset/coco/mass_train.json"
VAL_JSON = "/content/drive/MyDrive/EMBED_Dataset/curated/full_dataset/coco/mass_val.json"
DINOV2_WEIGHTS = "/content/drive/MyDrive/EMBED_Dataset/checkpoints/dinov2_latest_checkpoint.pth"
OUTPUT_DIR = "/content/drive/MyDrive/EMBED_Dataset/curated/full_dataset/RF_DETR"

print(f"Data Dir: {DATA_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")

In [ ]:
# Step 4: Visualize Dataset Annotations (Optional check)
!python visualize.py \
    --train-json "$TRAIN_JSON" \
    --val-json "$VAL_JSON" \
    --images-dir "$DATA_DIR" \
    --split train \
    --num-images 3 \
    --save-dir "$OUTPUT_DIR/viz"

In [ ]:
# Step 5: Launch Training via CLI
# Effective Batch Size = 2 * 8 = 16 (grad accumulation)
# Mixed Precision: auto-detected (bf16 on A100/L4, fp16 on T4/V100)
!python train.py \
    --config-file configs/mammo_dinov2_dino.py \
    --train-json "$TRAIN_JSON" \
    --val-json "$VAL_JSON" \
    --images-dir "$DATA_DIR" \
    --dinov2-weights "$DINOV2_WEIGHTS" \
    --output-dir "$OUTPUT_DIR" \
    --accum-steps 8 \
    --amp-dtype auto \
    --num-gpus 1


In [ ]:
# Step 6: Launch TensorBoard Monitoring
%load_ext tensorboard
%tensorboard --logdir "$OUTPUT_DIR"

In [ ]:
# Step 7: Standalone COCO Evaluation on Best Model
CHECKPOINT_PATH = f"{OUTPUT_DIR}/model_best.pth"

!python eval.py \
    --config-file configs/mammo_dinov2_dino.py \
    --weights "$CHECKPOINT_PATH" \
    --val-json "$VAL_JSON" \
    --images-dir "$DATA_DIR" \
    --output-dir "$OUTPUT_DIR/eval_results"

In [ ]:
# Step 8: Side-by-Side Comparison (Ground Truth vs Model Predictions)
!python visualize.py \
    --weights "$CHECKPOINT_PATH" \
    --config-file configs/mammo_dinov2_dino.py \
    --val-json "$VAL_JSON" \
    --images-dir "$DATA_DIR" \
    --conf-thresh 0.30 \
    --num-images 4 \
    --save-dir "$OUTPUT_DIR/predictions_viz"

# Display the generated comparison figure directly in the notebook
from IPython.display import Image, display
import os
viz_img = f"{OUTPUT_DIR}/predictions_viz/pred_vs_gt_mammo_val.png"
if os.path.exists(viz_img):
    display(Image(viz_img))